In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
!pip -q install catboost lightgbm

import numpy as np
import pandas as pd

from sklearn.metrics import roc_auc_score, average_precision_score

PATH = "/content/drive/MyDrive/HI-Small_Trans.csv"  # adjust if needed
df = pd.read_csv(PATH)

# Columns (your dataset)
TIME_COL = "Timestamp"
SRC_BANK = "From Bank"
SRC_ACC  = "Account"
DST_BANK = "To Bank"
DST_ACC  = "Account.1"
AMT_REC  = "Amount Received"
RCV_CUR  = "Receiving Currency"
AMT_PAY  = "Amount Paid"
PAY_CUR  = "Payment Currency"
PAY_FMT  = "Payment Format"
LBL_COL  = "Is Laundering"

# Clean
df[LBL_COL] = df[LBL_COL].astype(int)
df[AMT_PAY] = pd.to_numeric(df[AMT_PAY], errors="coerce").fillna(0.0)
df[AMT_REC] = pd.to_numeric(df[AMT_REC], errors="coerce").fillna(0.0)

df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
df = df.sort_values(TIME_COL).reset_index(drop=True)

# ---- Node aggregates (NON-GNN, strong signal) ----
# outgoing features per sender account
out_grp = df.groupby(SRC_ACC)[AMT_PAY]
df["src_out_deg"]  = out_grp.transform("size").astype(np.int32)
df["src_out_sum"]  = out_grp.transform("sum").astype(np.float32)
df["src_out_mean"] = out_grp.transform("mean").astype(np.float32)

# incoming features per receiver account
in_grp = df.groupby(DST_ACC)[AMT_PAY]
df["dst_in_deg"]  = in_grp.transform("size").astype(np.int32)
df["dst_in_sum"]  = in_grp.transform("sum").astype(np.float32)
df["dst_in_mean"] = in_grp.transform("mean").astype(np.float32)

# numeric edge features
df["log_amt_paid"] = np.log1p(df[AMT_PAY].astype(np.float32))
df["log_amt_recv"] = np.log1p(df[AMT_REC].astype(np.float32))
df["cross_bank"]   = (df[SRC_BANK].astype(str) != df[DST_BANK].astype(str)).astype(np.int8)

# Feature set
cat_cols = [SRC_BANK, DST_BANK, RCV_CUR, PAY_CUR, PAY_FMT]
num_cols = [
    "log_amt_paid", "log_amt_recv", "cross_bank",
    "src_out_deg", "src_out_sum", "src_out_mean",
    "dst_in_deg", "dst_in_sum", "dst_in_mean",
]

X = df[cat_cols + num_cols].copy()
y = df[LBL_COL].to_numpy(np.int32)

# Time-based split (70/15/15)
n = len(df)
n_train = int(0.7 * n)
n_val   = int(0.15 * n)

X_train, y_train = X.iloc[:n_train], y[:n_train]
X_val,   y_val   = X.iloc[n_train:n_train+n_val], y[n_train:n_train+n_val]
X_test,  y_test  = X.iloc[n_train+n_val:], y[n_train+n_val:]

print("Shapes:", X_train.shape, X_val.shape, X_test.shape)
print("Pos rate:", y_train.mean())


Shapes: (3554841, 14) (761751, 14) (761753, 14)
Pos rate: 0.0008034114605969719


In [ ]:
from catboost import CatBoostClassifier

cat_idx = [X_train.columns.get_loc(c) for c in cat_cols]

model_cb = CatBoostClassifier(
    iterations=3000,
    learning_rate=0.05,
    depth=8,
    loss_function="Logloss",
    eval_metric="AUC",
    random_seed=42,
    verbose=200,
    # imbalance handling
    auto_class_weights="Balanced",
)

model_cb.fit(
    X_train, y_train,
    cat_features=cat_idx,
    eval_set=(X_val, y_val),
    use_best_model=True
)

p_val  = model_cb.predict_proba(X_val)[:, 1]
p_test = model_cb.predict_proba(X_test)[:, 1]

print("\n=== CatBoost ===")
print("VAL  ROC:", roc_auc_score(y_val, p_val))
print("VAL  PR :", average_precision_score(y_val, p_val))
print("TEST ROC:", roc_auc_score(y_test, p_test))
print("TEST PR :", average_precision_score(y_test, p_test))


0:	test: 0.9201504	best: 0.9201504 (0)	total: 788ms	remaining: 39m 22s
200:	test: 0.9816024	best: 0.9816024 (200)	total: 4m 1s	remaining: 55m 57s
400:	test: 0.9814057	best: 0.9819275 (330)	total: 8m 22s	remaining: 54m 17s
600:	test: 0.9809728	best: 0.9819275 (330)	total: 13m 9s	remaining: 52m 33s
800:	test: 0.9808464	best: 0.9819275 (330)	total: 17m 57s	remaining: 49m 18s
1000:	test: 0.9804065	best: 0.9819275 (330)	total: 22m 44s	remaining: 45m 23s
1200:	test: 0.9800866	best: 0.9819275 (330)	total: 27m 36s	remaining: 41m 21s
1400:	test: 0.9799153	best: 0.9819275 (330)	total: 32m 30s	remaining: 37m 5s
1600:	test: 0.9796794	best: 0.9819275 (330)	total: 37m 29s	remaining: 32m 45s
1800:	test: 0.9794186	best: 0.9819275 (330)	total: 42m 24s	remaining: 28m 14s
2000:	test: 0.9792830	best: 0.9819275 (330)	total: 46m 50s	remaining: 23m 23s
2200:	test: 0.9792306	best: 0.9819275 (330)	total: 51m 15s	remaining: 18m 36s
2400:	test: 0.9790479	best: 0.9819275 (330)	total: 55m 42s	remaining: 13m 53s
26

In [ ]:
import lightgbm as lgb

X_train_lgb = X_train.copy()
X_val_lgb   = X_val.copy()
X_test_lgb  = X_test.copy()

# LightGBM needs pandas category dtype for native categorical handling
for c in cat_cols:
    X_train_lgb[c] = X_train_lgb[c].astype("category")
    X_val_lgb[c]   = X_val_lgb[c].astype("category")
    X_test_lgb[c]  = X_test_lgb[c].astype("category")

pos = y_train.mean()
scale_pos_weight = (1 - pos) / max(pos, 1e-9)

model_lgb = lgb.LGBMClassifier(
    n_estimators=5000,
    learning_rate=0.03,
    num_leaves=64,
    max_depth=-1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    objective="binary",
    n_jobs=-1,
    random_state=42,
    scale_pos_weight=scale_pos_weight,
)

model_lgb.fit(
    X_train_lgb, y_train,
    eval_set=[(X_val_lgb, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(stopping_rounds=200), lgb.log_evaluation(200)],
    categorical_feature=cat_cols
)

p_val  = model_lgb.predict_proba(X_val_lgb)[:, 1]
p_test = model_lgb.predict_proba(X_test_lgb)[:, 1]

print("\n=== LightGBM ===")
print("VAL  ROC:", roc_auc_score(y_val, p_val))
print("VAL  PR :", average_precision_score(y_val, p_val))
print("TEST ROC:", roc_auc_score(y_test, p_test))
print("TEST PR :", average_precision_score(y_test, p_test))


[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] Number of positive: 2856, number of negative: 3551985
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.127689 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 5396
[LightGBM] [Info] Number of data points in the train set: 3554841, number of used features: 14
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.000803 -> initscore=-7.125840
[LightGBM] [Info] Start training from score -7.125840
Training until validation scores don't improve for 200 rounds
[200]	valid_0's auc: 0.858945	valid_0's binary_log